<a href="https://colab.research.google.com/github/FOYEJ1027/-Neural-Networks-Lab/blob/main/Transfer_Learning_with_ResNet_18_on_CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Name : MD MOSADDEK HASAN FOYEJ ID : 0432310005101027 (A2)


  Transfer Learning with ResNet-18 on CIFAR-10

### Cell 1: Setup

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.CIFAR10('data', train=True, download=True, transform=tf)
test_ds = datasets.CIFAR10('data', train=False, download=True, transform=tf)

train_dl = DataLoader(train_ds, 64, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, 128)

device = 'cuda' if torch.cuda.is_available() else 'cpu'


### Cell 2: Part A, Feature Extraction

In [ ]:
# TODO (1): Load a ResNet-18 pretrained on ImageNet.
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# TODO (2): Freeze every parameter.
for param in model.parameters():
    param.requires_grad = False

# TODO (3): Replace the final layer with 10 output classes.
model.fc = nn.Linear(model.fc.in_features, 10)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

# TODO (4): Create optimizer for only the new fc layer.
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(model(xb), yb).backward()
        optimizer.step()

model.eval()
correct = 0

with torch.no_grad():
    for xb, yb in test_dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()

print(f'Feature-extraction accuracy: {100*correct/len(test_ds):.2f}%')


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


### Cell 3: Part B, Full Fine-Tuning

In [ ]:
# TODO (5): Unfreeze the entire network.
for param in model.parameters():
    param.requires_grad = True

# TODO (6): Create optimizer for the whole network with smaller learning rate.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# TODO (7): Train and evaluate the model.
for epoch in range(3):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(model(xb), yb).backward()
        optimizer.step()

model.eval()
correct = 0

with torch.no_grad():
    for xb, yb in test_dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()

print(f'Fine-tuning accuracy: {100*correct/len(test_ds):.2f}%')
